In [3]:
import pandas as pd
import cbbpy.mens_scraper as s
import boto3
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from scipy.stats import norm

import sys
sys.path.append("..")
from utils.clean_names import normalize_team_name

In [4]:
today = datetime.now(ZoneInfo("America/New_York"))
todays_games_ids = s.get_game_ids(date=today)

games = []

for i, game_id in enumerate(todays_games_ids, start=1):
    try:
        game, _, _ = s.get_game(game_id, box = False, pbp = False)
        games.append(game)
        print(f'Successfully recieved {game_id}! - {i} / {len(todays_games_ids)}')
    except Exception as e:
        print(f"Failed to retrieve {game_id}: {e}")

schedule = pd.concat(games, ignore_index=True)

Successfully recieved 401822972! - 1 / 37
Successfully recieved 401814601! - 2 / 37
Successfully recieved 401851460! - 3 / 37
Successfully recieved 401851701! - 4 / 37
Successfully recieved 401851749! - 5 / 37
Successfully recieved 401851624! - 6 / 37
Successfully recieved 401851350! - 7 / 37
Successfully recieved 401823787! - 8 / 37
Successfully recieved 401851347! - 9 / 37
Successfully recieved 401851245! - 10 / 37
Successfully recieved 401851488! - 11 / 37
Successfully recieved 401856422! - 12 / 37
Successfully recieved 401851348! - 13 / 37
Successfully recieved 401851246! - 14 / 37
Successfully recieved 401851489! - 15 / 37
Successfully recieved 401851579! - 16 / 37
Successfully recieved 401851247! - 17 / 37
Successfully recieved 401851700! - 18 / 37
Successfully recieved 401851457! - 19 / 37
Successfully recieved 401851349! - 20 / 37
Successfully recieved 401814602! - 21 / 37
Successfully recieved 401814598! - 22 / 37
Successfully recieved 401851621! - 23 / 37
Successfully recieve

In [5]:
cols = ['home_team', 'away_team', 'is_neutral', 'game_time', 'tv_network']
schedule = schedule[cols]
schedule['home_team'] = schedule['home_team'].apply(normalize_team_name)
schedule['away_team'] = schedule['away_team'].apply(normalize_team_name)

In [7]:
current_rankings = pd.read_csv('../data/current_rankings.csv')
current_rankings = current_rankings[current_rankings['Date'] == '2026-03-01']

team_rating_map = current_rankings.set_index('Team')['Total']
schedule['home_team_rating'] = schedule['home_team'].map(team_rating_map)
schedule['away_team_rating'] = schedule['away_team'].map(team_rating_map)

In [8]:
def home_team_spread(home, away, neutral):
    spread = (home - away) * 0.679
    spread += (~neutral) * 2.9
    return spread

def home_team_wp(home_spread):
    home_wp = 1 - norm.cdf(0, loc=home_spread, scale=10)
    away_wp = 1 - home_wp
    return home_wp

In [9]:
schedule['home_team_spread'] = home_team_spread(schedule['home_team_rating'], schedule['away_team_rating'], schedule['is_neutral'])
schedule['home_team_wp'] = home_team_wp(schedule['home_team_spread'])
schedule['away_team_wp'] = 1 - schedule['home_team_wp']
schedule['home_team_spread'] = schedule['home_team_spread'] * -1

schedule[['home_team_wp', 'away_team_wp']] = (schedule[['home_team_wp', 'away_team_wp']] * 100).round(2)
schedule['home_team_spread'] = schedule['home_team_spread'].round(1)

In [ ]:
schedule['home_team_spread'] = schedule['home_team_spread'].astype(str) + schedule['home_team']